# n_mels=128 Yeniden Koşum — Ortam Karışıklığını Kaldırmak İçin

**Neden:** Makaledeki 2x3 faktöriyel ablasyonda `n_mels=64` hücreleri bu Colab
ortamında (librosa 0.11.0 / TF 2.20.0) koştu, `n_mels=128` hücreleri ise aylar
önceki ana ızgaradan geliyor. Bu yüzden `n_mels` **ana etkisi** çalışma ortamıyla
karışmış durumda.

**Ne yapar:** `n_mels = 128`, `hop_length = 256`, üç `n_fft` (512/1024/2048),
beş tohum — yani ablasyonun eksik kolunu **aynı ortamda** yeniden üretir.
Mimari, hiperparametreler, tohumlar ve veri bölmesi değişmez.

**İki kazanç:**
1. 2x3 faktöriyel artık tek bir ortamda; `n_mels` ana etkisi ve etkileşim
   karışıklıksız yorumlanabilir.
2. Yeni `n_mels=128` sonuçları eski ana ızgara sonuçlarıyla karşılaştırılarak
   **ortam etkisinin büyüklüğü ölçülür** — bu kendi başına raporlanabilir.

**Süre:** ~1 saat (öznitelik çıkarımı 3 x ~2 dk + 15 eğitim koşusu).

**Çıktı:** `GTZAN PAPER/ablation_results_mel128/ablation_all_results.json`

> Mevcut `ablation_results/` klasörüne DOKUNMAZ; ayrı klasöre yazar.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================
# ROOT'U OTOMATIK BUL
# Drive nereye baglanmis olursa olsun (MyDrive, Shared drive, kisayol)
# "GTZAN PAPER" klasorunu isaretci dosyadan bulur.
# =============================================================
import os, glob

MARKER = os.path.join("extracted_features_with_segments_new",
                      "fft1024_hop256_mel128_seg3", "segments.json")

CANDIDATES = [
    "/content/drive/MyDrive/GTZAN PAPER",
    "/content/drive/MyDrive/Tüm Dosyalar/Papers/GTZAN PAPER",
    "/content/drive/MyDrive/Papers/GTZAN PAPER",
]
CANDIDATES += glob.glob("/content/drive/Shareddrives/*/Papers/GTZAN PAPER")
CANDIDATES += glob.glob("/content/drive/Shareddrives/*/*/Papers/GTZAN PAPER")
CANDIDATES += glob.glob("/content/drive/Shareddrives/*/GTZAN PAPER")

ROOT = None
for c in CANDIDATES:
    if os.path.exists(os.path.join(c, MARKER)):
        ROOT = c; break

if ROOT is None:                      # son care: sinirli derinlikte tara
    print("Aday yollarda bulunamadi, Drive taraniyor (biraz surebilir)...")
    for base in ("/content/drive/MyDrive", "/content/drive/Shareddrives"):
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, _ in os.walk(base):
            if dirpath.count(os.sep) - base.count(os.sep) > 4:
                dirnames[:] = []      # 4 seviyeden derine inme
                continue
            dirnames[:] = [d for d in dirnames
                           if not d.startswith(".") and "genres_original" not in d]
            if os.path.basename(dirpath) == "GTZAN PAPER" \
               and os.path.exists(os.path.join(dirpath, MARKER)):
                ROOT = dirpath; break
        if ROOT: break

assert ROOT, ("GTZAN PAPER klasoru bulunamadi. Drive'da 'extracted_features_with_segments_new/"
              "fft1024_hop256_mel128_seg3/segments.json' dosyasini iceren klasorun yolunu "
              "elle ROOT degiskenine yaz.")
print("ROOT bulundu:", ROOT)


In [ ]:
# =============================================================
# AYARLAR
# =============================================================
import os, json, time, gc
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# ROOT bir onceki hucrede otomatik bulundu
AUDIO_DIR  = os.path.join(ROOT, "GTZAN DATA", "genres_original")
MASTER     = os.path.join(ROOT, "extracted_features_with_segments_new",
                          "fft1024_hop256_mel128_seg3")
OUT_ROOT   = os.path.join(ROOT, "ablation_results_mel128")
os.makedirs(OUT_ROOT, exist_ok=True)

SR = 22050; TRACK_DURATION = 30; SEGMENT_DURATION = 3
N_MELS  = 128        # <-- ana izgarayla ayni; ortam karisikligini kaldirmak icin
HOP     = 256
N_FFTS  = [512, 1024, 2048]
SEEDS   = [42, 123, 456, 789, 1024]
BATCH_SIZE = 32
EPOCHS     = 120          # orijinal kodla ayni

print("audio :", os.path.exists(AUDIO_DIR))
print("master:", os.path.exists(MASTER))
import sys
print("tf    :", tf.__version__, "| librosa:", librosa.__version__)
print("python:", sys.version.split()[0], "| numpy:", np.__version__)
print("GPU   :", tf.config.list_physical_devices("GPU"))

In [ ]:
# =============================================================
# ORIJINAL ON-UC FONKSIYONLARI (Feature_Extraction.ipynb ile birebir ayni)
# =============================================================
def load_full_audio(file_path, sr=22050, track_duration=30):
    y, _ = librosa.load(file_path, sr=sr, mono=True)
    target_length = int(sr * track_duration)
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)), mode="constant")
    else:
        y = y[:target_length]
    return y

def split_into_segments(y, sr=22050, segment_duration=3):
    segment_length = int(sr * segment_duration)
    return [y[i*segment_length:(i+1)*segment_length]
            for i in range(len(y) // segment_length)]

def extract_log_mel_spectrogram(y, sr=22050, n_fft=1024, hop_length=256, n_mels=128):
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length,
        win_length=n_fft, n_mels=n_mels, power=2.0)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    mean = np.mean(log_mel); std = np.std(log_mel)
    if std < 1e-8: std = 1e-8
    log_mel = (log_mel - mean) / std
    return np.expand_dims(log_mel, axis=-1).astype(np.float32)

In [ ]:
# =============================================================
# ANA BOLME VE PARCA SIRASI — orijinal segments.json'dan okunur
# Boylece uretilen X, mevcut train/val/test indeksleriyle birebir hizalanir.
# =============================================================
segs = json.load(open(os.path.join(MASTER, "segments.json")))
y_all = np.load(os.path.join(MASTER, "y.npy"))
SP = os.path.join(MASTER, "track_level_split")
train_idx = np.load(os.path.join(SP, "train_idx.npy"))
val_idx   = np.load(os.path.join(SP, "val_idx.npy"))
test_idx  = np.load(os.path.join(SP, "test_idx.npy"))

seg_rel = ["/".join(s["track_path"].split("/")[-2:]) for s in segs]   # genre/file.wav
tracks  = list(dict.fromkeys(seg_rel))

print(f"segment {len(segs)} | parca {len(tracks)}")
print(f"bolme  train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)}")
assert len(segs) == len(train_idx) + len(val_idx) + len(test_idx)

# test segmentleri parca basina 10'luk ardisik bloklar halinde
test_track_indices = np.repeat(np.arange(len(test_idx)//10), 10)
y_train = y_all[train_idx]; y_val = y_all[val_idx]; y_test = y_all[test_idx]

In [ ]:
# =============================================================
# SESI BIR KEZ BELLEGE AL (~2.6 GB) — 3 konfigurasyon icin tekrar okunmaz
# =============================================================
t0 = time.time()
AUDIO = {}
for i, t in enumerate(tracks):
    AUDIO[t] = load_full_audio(os.path.join(AUDIO_DIR, t), SR, TRACK_DURATION)
    if (i+1) % 200 == 0:
        print(f"  {i+1}/{len(tracks)}  {time.time()-t0:.0f}s")
print(f"ses yuklendi: {len(AUDIO)} parca, {time.time()-t0:.0f}s, "
      f"~{sum(a.nbytes for a in AUDIO.values())/1e9:.2f} GB")

In [ ]:
# =============================================================
# MODEL — orijinal mimarinin birebir aynisi
# =============================================================
def build_model(input_shape, num_classes=10):
    REG = regularizers.l2(5e-4)
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3,3), padding="same", activation="relu", kernel_regularizer=REG),
        layers.BatchNormalization(), layers.SpatialDropout2D(0.15), layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), padding="same", activation="relu", kernel_regularizer=REG),
        layers.BatchNormalization(), layers.SpatialDropout2D(0.2),  layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), padding="same", activation="relu", kernel_regularizer=REG),
        layers.BatchNormalization(), layers.SpatialDropout2D(0.25), layers.MaxPooling2D((2,2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation="relu", kernel_regularizer=REG),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                  metrics=["accuracy"])
    return model

def majority_vote(y_true, y_pred, track_indices):
    tt, tp = [], []
    for tid in np.unique(track_indices):
        m = track_indices == tid
        tt.append(y_true[m][0])
        tp.append(Counter(y_pred[m].tolist()).most_common(1)[0][0])
    return np.array(tt), np.array(tp)

def soft_aggregate(probs, track_indices):
    return np.array([int(probs[track_indices == tid].mean(0).argmax())
                     for tid in np.unique(track_indices)])

In [ ]:
# =============================================================
# ABLASYON — 3 konfigurasyon x 5 tohum
# =============================================================
all_runs = []

for n_fft in N_FFTS:
    name = f"fft{n_fft}_hop{HOP}_mel{N_MELS}_seg3"
    print("\n" + "="*72); print(f"  {name}"); print("="*72)

    # --- oznitelik cikarimi (bellekte) ---
    t0 = time.time()
    W = 1 + (SR*SEGMENT_DURATION)//HOP
    X = np.zeros((len(segs), N_MELS, W, 1), dtype=np.float32)
    pos = 0
    for t in tracks:
        for seg in split_into_segments(AUDIO[t], SR, SEGMENT_DURATION):
            X[pos] = extract_log_mel_spectrogram(seg, SR, n_fft, HOP, N_MELS)
            pos += 1
    assert pos == len(segs), (pos, len(segs))
    print(f"  oznitelik: {X.shape}  {time.time()-t0:.0f}s")

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    del X; gc.collect()

    input_shape = X_train.shape[1:]
    y_train_oh = tf.keras.utils.to_categorical(y_train, 10)
    y_val_oh   = tf.keras.utils.to_categorical(y_val, 10)
    y_test_oh  = tf.keras.utils.to_categorical(y_test, 10)

    cfg_dir = os.path.join(OUT_ROOT, name); os.makedirs(cfg_dir, exist_ok=True)
    runs = []

    for run_idx, seed in enumerate(SEEDS):
        seed_dir = os.path.join(cfg_dir, f"seed_{seed}"); os.makedirs(seed_dir, exist_ok=True)
        rpath = os.path.join(seed_dir, "result.json")
        if os.path.exists(rpath):
            print(f"  seed {seed}: zaten var, atlaniyor"); runs.append(json.load(open(rpath))); continue

        print(f"\n  --- seed {seed} ({run_idx+1}/{len(SEEDS)}) ---")
        tf.keras.utils.set_random_seed(seed)
        train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train_oh))
                    .shuffle(len(X_train), seed=seed).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
        val_ds  = tf.data.Dataset.from_tensor_slices((X_val,  y_val_oh )).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test_oh)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        model = build_model(input_shape)
        callbacks = [
            EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
            ModelCheckpoint(os.path.join(seed_dir, "best_model.keras"),
                            monitor="val_loss", save_best_only=True, verbose=0),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        ]
        t0 = time.time()
        hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                         callbacks=callbacks, verbose=2)
        train_time = time.time() - t0

        test_loss, seg_acc = model.evaluate(test_ds, verbose=0)
        probs = model.predict(test_ds, verbose=0)
        y_pred_seg = probs.argmax(1)
        yt_track, yp_hard = majority_vote(y_test, y_pred_seg, test_track_indices)
        yp_soft = soft_aggregate(probs, test_track_indices)
        be = int(np.argmin(hist.history["val_loss"]))

        res = {
            "config": name, "seed": seed, "run_idx": run_idx,
            "fft": n_fft, "hop": HOP, "n_mels": N_MELS,
            "input_shape": list(input_shape), "time_frames": int(input_shape[1]),
            "total_params": int(model.count_params()),
            "best_epoch": be + 1, "total_epochs": len(hist.history["loss"]),
            "train_time_sec": round(train_time, 1),
            "train_acc": float(hist.history["accuracy"][be]),
            "val_acc": float(hist.history["val_accuracy"][be]),
            "overfit_gap": float(hist.history["accuracy"][be] - hist.history["val_accuracy"][be]),
            "seg_test_acc": float(seg_acc), "seg_test_loss": float(test_loss),
            "seg_f1_macro": float(f1_score(y_test, y_pred_seg, average="macro")),
            "seg_f1_per_class": f1_score(y_test, y_pred_seg, average=None).tolist(),
            "seg_confusion_matrix": confusion_matrix(y_test, y_pred_seg).tolist(),
            "track_test_acc": float(accuracy_score(yt_track, yp_hard)),
            "track_f1_macro": float(f1_score(yt_track, yp_hard, average="macro")),
            "track_f1_per_class": f1_score(yt_track, yp_hard, average=None).tolist(),
            "track_confusion_matrix": confusion_matrix(yt_track, yp_hard).tolist(),
            "track_test_acc_soft": float(accuracy_score(yt_track, yp_soft)),
            "track_f1_macro_soft": float(f1_score(yt_track, yp_soft, average="macro")),
            "history": {k: [float(x) for x in v] for k, v in hist.history.items()
                        if k in ("accuracy","val_accuracy","loss","val_loss")},
        }
        json.dump(res, open(rpath, "w"), indent=2)
        np.save(os.path.join(seed_dir, "test_probs.npy"), probs.astype(np.float32))
        runs.append(res)
        print(f"  >> seg {seg_acc:.4f} | parca(sert) {res['track_test_acc']:.4f} "
              f"| parca(yumusak) {res['track_test_acc_soft']:.4f} | {train_time:.0f}s")
        del model, probs; tf.keras.backend.clear_session(); gc.collect()

    json.dump(runs, open(os.path.join(cfg_dir, "all_seeds_results.json"), "w"), indent=2)
    all_runs += runs
    del X_train, X_val, X_test; gc.collect()

json.dump(all_runs, open(os.path.join(OUT_ROOT, "ablation_all_results.json"), "w"), indent=2)
print("\nTAMAMLANDI ->", os.path.join(OUT_ROOT, "ablation_all_results.json"))

In [ ]:
# =============================================================
# ORTAM ETKISININ OLCUMU
# yeni n_mels=128 (bu ortam) vs eski n_mels=128 (ana izgara)
# =============================================================
# Ana izgaradan, hop=256, n_mels=128 (makaledeki Tablo 6 degerleri):
old128 = {512: (0.7843, 0.8573, 0.8720),
          1024: (0.7921, 0.8600, 0.8800),
          2048: (0.7991, 0.8533, 0.8720)}   # (segment, parca sert, parca yumusak)

print(f"{'n_fft':>6} | {'seg yeni':>9} {'seg eski':>9} {'fark(pp)':>9}"
      f" | {'sert yeni':>10} {'sert eski':>10} {'fark(pp)':>9}"
      f" | {'yum. yeni':>10} {'yum. eski':>10} {'fark(pp)':>9}")
print("-" * 116)
rows = []
for n_fft in N_FFTS:
    r = [x for x in all_runs if x["fft"] == n_fft]
    sg = float(np.mean([x["seg_test_acc"] for x in r]))
    hd = float(np.mean([x["track_test_acc"] for x in r]))
    sf = float(np.mean([x["track_test_acc_soft"] for x in r]))
    o_sg, o_hd, o_sf = old128[n_fft]
    rows.append((n_fft, sg, hd, sf))
    print(f"{n_fft:>6} | {sg:>9.4f} {o_sg:>9.4f} {100*(sg-o_sg):>+9.2f}"
          f" | {hd:>10.4f} {o_hd:>10.4f} {100*(hd-o_hd):>+9.2f}"
          f" | {sf:>10.4f} {o_sf:>10.4f} {100*(sf-o_sf):>+9.2f}")

d = [100*(r[1]-old128[r[0]][0]) for r in rows]
print(f"\nSegment duzeyi ortalama kayma: {np.mean(d):+.2f} pp"
      f"  (aralik {min(d):+.2f} .. {max(d):+.2f})")
print("Kayma kucukse (|.| < ~0.5 pp) ortam etkisi ihmal edilebilir demektir.")
print("Kayma buyukse ana izgara sonuclari bu ortamda tekrarlanmiyor -> raporlanmali.")

# n_fft egilimi bu ortamda da artan mi? (mel=128 kolunda beklenen: artan)
print(f"\nBu ortamda n_mels=128 kolunda n_fft egilimi: "
      f"{rows[0][1]:.4f} -> {rows[1][1]:.4f} -> {rows[2][1]:.4f}")
print("Makaledeki mel=64 kolu (azalan): 0.7911 -> 0.7832 -> 0.7748")
print("Iki kol ters yonde kaliyorsa etkileşim bulgusu ayni ortamda dogrulanmis olur.")
